# Práctica de Laboratorio 1 
# Parte 1: Análisis Exploratorio de Datos

### Información del Grupo
* **Grupo de Laboratorio:** Viernes a las 08:00h
* **Número de Grupo:** Equipo 1
* **Integrantes:**
  * Lucas Romero Domingo
  * Marco Alcalde Campos
  * Unai Herrero Etxeberria
* **Incidencias:** Ninguna reseñable; participación totalmente equitativa y coordinada entre los miembros.

# Presentación del problema y de los datos

### Procedencia y contexto
* **Fuente de los datos:** Conjunto de datos sobre especificaciones y precios de ordenadores portátiles obtenido del repositorio público Kaggle ([Laptop Prices Dataset](https://www.kaggle.com/code/a7med3laa7/laptop-prices/input)).
* **Unidad de observación:** Cada registro representa un modelo específico de ordenador portátil comercializado en el mercado.
* **Problema a estudiar:** La industria de los ordenadores portátiles ofrece una amplia variedad de gamas (equipos de oficina, ultraligeros, estaciones de trabajo o portátiles para videojuegos), lo que genera una notable dispersión de precios. El objetivo de este análisis exploratorio es evaluar cómo influyen las características físicas y de hardware en el coste final, así como estudiar el posicionamiento de mercado de los distintos fabricantes.

### Variables seleccionadas
Hemos seleccionado 6 variables representativas (4 cuantitativas y 2 cualitativas) que combinan especificaciones técnicas y atributos comerciales:

1. **Price** (*Numérica continua / razón*): Precio del portátil expresado en euros. Es la variable objetivo o de interés principal.
2. **Company** (*Categórica nominal*): Marca o fabricante del equipo (Dell, Lenovo, HP, Apple, Asus, etc.).
3. **TypeName** (*Categórica nominal*): Tipología o categoría de diseño del dispositivo (Notebook, Ultrabook, Gaming, 2 in 1 Convertible, Workstation, Netbook).
4. **RAM** (*Numérica discreta*): Capacidad de memoria principal en gigabytes (GB). Componente crítico ligado al rendimiento multitarea y al coste del equipo.
5. **Weight** (*Numérica continua*): Peso del dispositivo medido en kilogramos (kg). Permite estudiar el compromiso entre portabilidad física y potencia.
6. **Inches** (*Numérica continua*): Diagonal de la pantalla medida en pulgadas. Determina el factor de forma y tamaño.

### Justificación del conjunto de variables
La elección combinada de estas seis variables responde a la necesidad de analizar el precio desde dos ángulos complementarios: el hardware interno y el formato físico del equipo. Por un lado, variables cuantitativas como `RAM` y `Weight` permiten cuantificar el coste directo de la potencia de procesamiento y el esfuerzo de ingeniería. Por otro lado, `Company` y `TypeName` capturan el valor de marca y el segmento objetivo (por ejemplo, el sobrecoste habitual de la gama profesional o *gaming* frente a la ofimática básica). Finalmente, `Inches` actúa como variable de escala que delimita el tamaño del ordenador y condiciona tanto la portabilidad como el tipo de uso previsto. En conjunto, este grupo reducido pero representativo permite responder a preguntas clave sobre qué atributos determinan en mayor medida el valor económico de un portátil sin introducir redundancias innecesarias.

## Inspección inicial y calidad de los datos

Antes de iniciar el estudio descriptivo y relacional, realizamos una auditoría inicial sobre el conjunto de datos para determinar sus dimensiones efectivas, comprobar la coherencia de los tipos de variables asignados y verificar la presencia de valores ausentes o registros duplicados. 

Asimismo, se detectan posibles inconsistencias de formato en el origen, cuya corrección resulta indispensable para posibilitar el cálculo posterior de estadísticos y representaciones gráficas.

In [6]:
import numpy as np
import pandas as pd

# Carga del dataset desde el archivo CSV
df_raw = pd.read_csv("data/laptop_prices.csv", encoding="latin-1")

print(f"Dimensiones completas del repositorio: {df_raw.shape[0]} observaciones y {df_raw.shape[1]} columnas.\n")

# Seleccionamos las variables de interés y las renombramos para facilitar su uso posterior
df = df_raw.copy()
if "Price_euros" in df.columns:
    df = df.rename(columns={"Price_euros": "Price"})
if "Ram" in df.columns:
    df = df.rename(columns={"Ram": "RAM"})

columnas_estudio = ["Company", "TypeName", "Inches", "RAM", "Weight", "Price"]
df_sub = df[columnas_estudio].copy()

# Estructura de las variables seleccionadas
df_sub.info()

# Recuento de valores nulos y porcentaje de valores nulos por variable
valores_nulos = df_sub.isnull().sum()
porcentaje_nulos = (valores_nulos / len(df_sub)) * 100
df_calidad = pd.DataFrame({"Valores Ausentes": valores_nulos, "Porcentaje (%)": porcentaje_nulos})
print(df_calidad)

# Detección de filas duplicadas
num_duplicados = df_sub.duplicated().sum()
print(f"\nNúmero de filas duplicadas en las 6 variables: {num_duplicados} ({(num_duplicados / len(df_sub)) * 100:.2f}%)")

# Comprobación de aspecto final del dataset
df_sub.head()

Dimensiones completas del repositorio: 1275 observaciones y 23 columnas.

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Company   1275 non-null   str    
 1   TypeName  1275 non-null   str    
 2   Inches    1275 non-null   float64
 3   RAM       1275 non-null   int64  
 4   Weight    1275 non-null   float64
 5   Price     1275 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 59.9 KB
          Valores Ausentes  Porcentaje (%)
Company                  0             0.0
TypeName                 0             0.0
Inches                   0             0.0
RAM                      0             0.0
Weight                   0             0.0
Price                    0             0.0

Número de filas duplicadas en las 6 variables: 15 (1.18%)


,Company,TypeName,Inches,RAM,Weight,Price
0,Apple,Ultrabook,13.3,8,1.37,1339.69
1,Apple,Ultrabook,13.3,8,1.34,898.94
2,HP,Notebook,15.6,8,1.86,575.00
3,Apple,Ultrabook,15.4,16,1.83,2537.45
4,Apple,Ultrabook,13.3,8,1.37,1803.60


### Diagnóstico de la calidad de los datos y observaciones relevantes

A la vista de los resultados obtenidos en la inspección inicial, se destacan los siguientes aspectos clave para las fases posteriores del estudio:

* **Adecuación muestral:** El subconjunto seleccionado cuenta con **1.275 observaciones** y **6 variables**. Esta dimensión cumple con creces el criterio orientativo de la práctica (entre unos centenares y pocos miles de registros), ofreciendo una muestra suficientemente amplia para obtener conclusiones estadísticas con validez empírica.
* **Coherencia de tipos de datos:** Las variables han quedado tipadas adecuadamente sin necesidad de transformaciones forzadas:
  * `Company` y `TypeName` como variables cualitativas (`str`).
  * `RAM` como variable cuantitativa discreta (`int64`).
  * `Inches`, `Weight` y `Price` como variables cuantitativas continuas (`float64`).
* **Completitud de los datos (ausencia de nulos):** Ninguna de las 6 columnas presenta valores ausentes o faltantes (`0.0%` de nulos). Esto evita la necesidad de recurrir a técnicas de imputación o de eliminar registros por pérdida de información, garantizando un análisis univariante y bivariante sobre el 100% de la muestra.
* **Registros duplicados:** Se identifican **15 filas repetidas** en el subconjunto de 6 variables (un `1.18%` de la muestra). En el sector de los ordenadores comerciales, este patrón es habitual y esperado: varios comercios o distribuidores ofertan configuraciones técnicas idénticas de serie (mismo fabricante, categoría, pantalla, memoria, peso y precio base). Al responder a la propia oferta comercial del mercado y no a un error en la toma de datos, se decide **conservar estos registros** para mantener la representatividad natural de los modelos más comunes.